# 00 — setup, data check, first baseline

A thin driver. Everything substantive lives in `src/mutstab/`, so it can be
tested and reused; this notebook calls into it and shows the output.

Order of business:

1. Mount Drive, clone the repo, install the package
2. Look at how concentrated the data is — this decides how to read every later number
3. Verify the split file does not leak
4. Run the properties-only baseline (no GPU, no embeddings)
5. Compare pooled against per-protein metrics

## 1. Environment

Re-run this cell after a reconnect. The clone is into `/content`, which is
disposable; nothing you care about should live there.

In [ ]:
import importlib.util
import os

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    REPO = "/content/mutstab"
    GIT_URL = "https://github.com/<you>/mutstab.git"  # <- your repo
    if not os.path.isdir(REPO):
        !git clone -q $GIT_URL $REPO
    else:
        !cd $REPO && git pull -q
    %pip install -q -e "$REPO[store]"
    os.chdir(REPO)
else:
    # Local: assume the notebook sits in <repo>/notebooks.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    os.environ.setdefault("MUTSTAB_DATA_ROOT", os.path.expanduser("~/data/mutstab"))

print("cwd:", os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from mutstab import paths, runs
from mutstab.config import load_config

paths.mount_drive()
print(paths.describe())

`autoreload` means edits to `src/` take effect without restarting the kernel.
Worth knowing: it does not reload module-level constants reliably, so restart
after changing one.

## 2. Data

Put the FireProtDB export in `<data_root>/raw/` and run
`scripts/prepare_data.py` once. The cell below loads the result.

In [ ]:
cfg = load_config("baseline")
print(cfg.run_id)

processed = paths.processed_dir() / cfg.get_path("data.processed_file")
if not processed.exists():
    raise SystemExit(
        f"no processed data at {processed}\n"
        "Run: python scripts/prepare_data.py --config baseline"
    )

# stage() copies off Drive onto local disk once — see paths.py on why.
df = pd.read_parquet(paths.stage(processed))
print(f"{len(df):,} rows, {df['protein_id'].nunique():,} proteins")
df.head()

### How biased is it?

Read this table before modelling. If a handful of proteins hold most of the
rows, a random split will flatter anything you build, and per-protein metrics
are the only ones worth quoting.

In [ ]:
from mutstab.data.load import summarise

table = summarise(df)
display(table.head(15))

top5 = table.head(5)["n"].sum() / len(df)
print(f"top 5 proteins: {top5:.0%} of all rows")
print(f"stabilising (dTm > 0): {(df['dtm'] > 0).mean():.1%}")
print(f"dTm mean {df['dtm'].mean():.2f}, sd {df['dtm'].std():.2f}")
print(
    "\nA low stabilising fraction is why 'predict negative' scores well and\n"
    "why reverse mutations are in the training config."
)

## 3. Splits

Generated once by `scripts/make_splits.py` and committed. This cell only
checks it — it never regenerates, because a split that changes underneath you
silently invalidates every earlier result.

In [ ]:
from mutstab.data.splits import assert_no_group_leakage, fold_summary, load_splits

spec = load_splits(cfg.get_path("split.file"))
assert_no_group_leakage(df, spec)  # raises if any protein is in two folds
print("split OK:", spec["_path"])
print(f"{spec['n_groups']} proteins, {spec['n_folds']} folds, "
      f"holdout {len(spec['holdout'])} proteins")

fold_summary(df, spec)

## 4. Properties-only baseline

No embeddings, no GPU, a few seconds. This is the number every later model has
to beat, and it is a less comfortable target than it looks.

In [ ]:
from mutstab.data.load import add_reverse_mutations
from mutstab.features.variant import build_feature_matrix
from mutstab.models.baseline import cross_validate

train_df = add_reverse_mutations(df) if cfg.get_path("data.add_reverse") else df.copy()

X, names, valid = build_feature_matrix(train_df, use_embeddings=False)
train_df = train_df.loc[valid].reset_index(drop=True)
print(f"X: {X.shape[0]:,} x {X.shape[1]}")

result = cross_validate(
    train_df,
    X,
    spec,
    {"kind": "ridge", "alpha": 10.0},
    on_fold=lambda f, m: print(
        f"  fold {f}: rmse {m['rmse']:.2f} (baseline {m['baseline_rmse']:.2f})"
    ),
)

## 5. Pooled vs per-protein

The comparison that tells you whether the model learned anything about
mutations, or just about which protein a row came from.

In [ ]:
pooled = result["pooled_oof"]
protein = result["per_protein_oof"]

print("pooled out-of-fold")
print(f"  n         {pooled['n']:,}")
print(f"  rmse      {pooled['rmse']:.2f}   (constant baseline {result['baseline_oof']['rmse']:.2f})")
print(f"  spearman  {pooled['spearman']}")

print(f"\nper protein ({protein['n_proteins']} with >= 5 measurements)")
print(f"  median spearman  {protein['median_spearman']}")
print(f"  IQR              {protein['iqr_spearman']}")
print(f"  frac positive    {protein['frac_positive_spearman']}")

print("\nacross folds (watch the sd — it is often large enough to")
print("make a small improvement in the mean meaningless)")
for key, value in sorted(result["across_folds"].items()):
    if key.startswith(("rmse", "spearman", "median_spearman")):
        print(f"  {key:34s} {value:.3f}" if isinstance(value, float) else f"  {key}: {value}")

In [ ]:
# Per-protein detail: where does it do well, and where does it fail?
pd.DataFrame(protein["per_protein"]).head(20)

### Antisymmetry

Physics requires `f(A->B) == -f(B->A)` exactly. A model trained only on
measured mutations usually violates it badly, which is a direct read on how
much it has simply learned "most mutations are destabilising".

In [ ]:
from mutstab.eval.metrics import antisymmetry_error
from mutstab.models.baseline import build_model

# Fit on everything, then ask for both directions of the same variants.
y = train_df["dtm"].to_numpy(dtype=float)
finite = np.isfinite(y)
model = build_model({"kind": "ridge", "alpha": 10.0}).fit(X[finite], y[finite])

forward_df = df.copy()
reverse_df = df.copy()
reverse_df["wt_aa"], reverse_df["mut_aa"] = df["mut_aa"].copy(), df["wt_aa"].copy()

Xf, _, vf = build_feature_matrix(forward_df, use_embeddings=False)
Xr, _, vr = build_feature_matrix(reverse_df, use_embeddings=False)
both = vf & vr

report = antisymmetry_error(
    model.predict(Xf[both[vf]]), model.predict(Xr[both[vr]])
)
print(report)
print(
    "\nrmse_antisymmetry near 0 is what you want. A clearly negative bias means\n"
    "the model calls both directions destabilising, which is impossible."
)

## Next

- `python scripts/embed.py --config baseline` on a GPU runtime, then
  `python scripts/train.py --config baseline` — does ESM beat properties alone?
- Add structure features (RSA, contact number) as `features.extra_columns`.
- Cluster sequences at 30% identity and pass cluster ids as the split groups;
  expect the numbers to drop, and trust those more.
- Try training on ddG with dTm as a fine-tuning target — there is far more ddG
  data, and the scarcity of dTm labels is the binding constraint here.

Compare runs with `runs.list_runs()`:

In [ ]:
records = runs.list_runs()
if records:
    display(
        pd.json_normalize(records)[
            [
                "run_id",
                "timestamp",
                "git_dirty",
                "metrics.pooled_oof.rmse",
                "metrics.pooled_oof.spearman",
                "metrics.per_protein_oof.median_spearman",
            ]
        ]
    )
else:
    print("no runs yet — scripts/train.py writes them to results/")